# V19 – Datenbanken Teil 1: Theorie

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- den Unterschied zwischen einer flachen Datei (CSV/JSON) und einer relationalen Datenbank konkret benennen,
- das **relationale Modell** (Tabelle, Zeile, Spalte, Schlüssel) in eigenen Worten erklären,
- die **SQL-Grundbefehle** `CREATE`, `INSERT`, `SELECT`, `UPDATE`, `DELETE` lesen und aufschreiben,
- **SQLite** einordnen und den Unterschied zu Server-DBs wie PostgreSQL oder MySQL skizzieren,
- das Python-Modul `sqlite3` mit **Parameter-Binding** korrekt und sicher gegen **SQL-Injection** einsetzen.

## ⏱️ Zeitbudget
Ca. 30 Minuten.

## 🧭 TL;DR
Eine **Datenbank** strukturiert Daten in **Tabellen** mit festen Spalten und ausdrucksstarken Abfragen. **SQLite** ist eine Datei-basierte Datenbank, die Python im Standardmodul `sqlite3` mitliefert. Die fünf wichtigsten SQL-Befehle sind `CREATE TABLE`, `INSERT`, `SELECT`, `UPDATE` und `DELETE`. Parameter-Binding mit `?` ist die einzige sichere Methode, Nutzereingaben in Queries einzubauen.

## 📦 Voraussetzungen
- [00_python_recap.ipynb](00_python_recap.ipynb).
- Grundverständnis von CSV-Dateien aus V09.


## 1. Warum überhaupt eine Datenbank?

Solange du einzelne Messreihen oder Konfigurations-Dateien verwaltest, reicht eine **CSV**- oder **JSON**-Datei häufig aus. Sobald du aber mehrere Tabellen verknüpfst, gleichzeitig von verschiedenen Programmen aus liest und schreibst oder konsistent auf Teilmengen zugreifst, stößt du an die Grenzen flacher Dateien sehr schnell.

Stell dir eine Fertigungshalle vor: Du erfasst **Maschinen**, die **Werkstücke** produzieren, zusätzlich **Wartungen** pro Maschine und **Sensormesswerte** im Sekundentakt. Eine einzelne CSV-Datei würde sich schnell aufblähen. Zwei oder drei CSV-Dateien ließen sich zwar organisieren, doch jeder Zugriff erfordert, die komplette Datei zu laden, zu filtern und anschließend wieder zu schreiben – ohne jede Garantie, dass zwei Programme sich nicht gegenseitig überschreiben.


### Konkrete Vorteile einer Datenbank

Eine **relationale Datenbank** bietet vier Eigenschaften, die flachen Dateien fehlen. Erstens garantiert sie **Konsistenz**: Zwei Zeilen mit identischer Produkt-ID werden durch einen **Primärschlüssel** verhindert, ein falsches Datumsformat durch eine `CHECK`-Bedingung abgefangen. Zweitens erlaubt sie **gleichzeitige Nutzung** durch mehrere Programme, ohne dass sich die Änderungen gegenseitig überschreiben. Drittens stellt sie **ausdrucksstarke Abfragen** zur Verfügung: „Alle Werkstücke, deren Produktion länger als 20 Minuten dauert und mehr als 2 kg Material benötigt" ist in SQL ein Einzeiler. Viertens lassen sich **Integritätsbedingungen** (z. B. „jeder Sensor muss zu einer existierenden Maschine gehören") direkt in der Datenbank erzwingen.

> [!NOTE]
> Eine **relationale Datenbank** speichert Daten in Form von **Relationen** (Tabellen) mit festen **Attributen** (Spalten) und erlaubt es, Beziehungen zwischen Tabellen über Schlüssel abzubilden.


## 2. Das relationale Modell

Das relationale Modell wurde 1970 von **Edgar F. Codd** beschrieben und hat bis heute die Praxis der meisten Datenbanken geprägt. Die zentralen Begriffe sind schnell erklärt, weil sie Alltagsvokabeln aus jeder Tabelle aufgreifen.

Eine **Tabelle** (formal: Relation) ist eine zweidimensionale Struktur mit einem festen **Schema**. Jede **Zeile** (auch **Datensatz** oder **Tupel**) beschreibt genau ein Objekt der realen Welt, etwa ein Produkt oder eine Messung. Jede **Spalte** (auch **Attribut**) legt einen Namen und einen Datentyp fest, zum Beispiel `Produktname TEXT` oder `Produktionszeit_Minuten INTEGER`.


In [1]:
# Diagramm: Struktur einer Tabelle
from IPython.display import Markdown, display
with open("diagramme/01_tabelle_struktur.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
classDiagram
    class Produkte {
        +INTEGER Produkt_ID  PK
        +TEXT Produktname  NOT NULL
        +INTEGER Produktionszeit_Minuten
        +REAL Material_pro_Stueck_kg
    }
    note for Produkte "Eine Zeile = ein Produkt. Spalten haben feste Datentypen."

```

### Primärschlüssel und Fremdschlüssel (Kurz-Version)

Ein **Primärschlüssel** ist eine Spalte (oder Spaltenkombination), die jede Zeile der Tabelle eindeutig identifiziert. In der Produkt-Tabelle wäre das die `Produkt_ID`. Ein **Fremdschlüssel** verweist von einer Tabelle auf den Primärschlüssel einer anderen und stellt so **Beziehungen** zwischen Tabellen her, etwa „dieser Sensor gehört zu dieser Maschine".

> [!NOTE]
> Ein **Primärschlüssel** (Primary Key) ist eine Spalte, deren Werte niemals `NULL` sein dürfen und pro Zeile eindeutig sein müssen. SQLite erzeugt für `INTEGER PRIMARY KEY`-Spalten automatisch fortlaufende IDs, falls du keinen Wert angibst.

Die Feinheiten von Fremdschlüsseln, Joins und Normalformen sind das Kernthema von **V20**. In V19 konzentrieren wir uns bewusst auf **eine einzelne Tabelle**, damit die SQL-Syntax im Vordergrund bleibt.


## 3. SQLite – Datenbank als Datei

**SQLite** ist eine Datenbank, die sich erheblich von den großen, bekannten Namen wie **PostgreSQL**, **MySQL** oder **Oracle** unterscheidet. Während diese Systeme als **Server-Prozesse** laufen, die Clients über ein Netzwerkprotokoll bedienen, ist SQLite eine reine **C-Bibliothek**, die eine einzige **Datei** (Endung `.db` oder `.sqlite`) als Datenbank behandelt.

> [!NOTE]
> **SQLite** ist eine Embedded-Datenbank ohne Server. Die komplette Datenbank liegt in einer einzigen Datei auf der Festplatte und wird direkt von der Anwendung über eine Bibliothek geöffnet.


In [2]:
# Architektur-Überblick
from IPython.display import Markdown, display
with open("diagramme/04_db_architektur.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart LR
    A["Python-Programm"] --> B["sqlite3-Modul\n(Standardbibliothek)"]
    B --> C["SQLite-Engine\n(C-Bibliothek)"]
    C --> D[("produktion.db\n(Datei auf der Platte)")]
    C --> E[(":memory:\n(RAM)")]

```

### Wann SQLite, wann ein Server-DBMS?

SQLite spielt seine Stärken überall dort aus, wo eine **eingebettete** Datenbank gebraucht wird: in Browsern, Smartphones, Desktop-Programmen, kleinen Web-Anwendungen mit wenigen gleichzeitigen Schreibern und – für uns besonders wichtig – in **Lernumgebungen** und kleinen Ingenieur-Tools. Jede laufende Android- oder iOS-App nutzt SQLite intern; viele Messgeräte und Steuerungssysteme schreiben ihre Daten ebenfalls in SQLite-Dateien.

Sobald jedoch **hunderte gleichzeitige Schreiber**, **strikte Zugriffsrechte** pro Benutzer oder **Replikation** über mehrere Rechner gefordert sind, sind Server-Datenbanken wie PostgreSQL oder MySQL die bessere Wahl. Der Umstieg ist nicht dramatisch, weil die SQL-Grundsyntax sehr ähnlich bleibt – die Konzepte dieser Vorlesung sind eins zu eins übertragbar.

> [!TIP]
> Faustregel: Für Lernen, Desktop-Tools, Prototypen und kleine Web-Apps ist **SQLite** erste Wahl. Für Mehrbenutzer-Systeme mit nennenswertem Schreib-Aufkommen nimmt man **PostgreSQL**.


### Die In-Memory-Datenbank

SQLite kann eine Datenbank auch komplett im Arbeitsspeicher halten. Der Verbindungsaufruf lautet dann `sqlite3.connect(":memory:")`. Das Ergebnis verhält sich wie eine normale Datenbank, verschwindet aber mit dem Programm wieder. Für Unit-Tests, Lern-Notebooks oder kurzlebige Analysen ist das ideal – und wir nutzen das in V19 durchgehend, damit dein Projektordner sauber bleibt.


## 4. SQL – die Sprache der Datenbanken

**SQL** (Structured Query Language) ist eine standardisierte Sprache, um relationale Datenbanken zu erstellen, zu befüllen und abzufragen. Ihre Befehle lassen sich grob in zwei Gruppen einteilen. Die **DDL** (Data Definition Language) definiert die Struktur der Datenbank: `CREATE TABLE`, `ALTER TABLE`, `DROP TABLE`. Die **DML** (Data Manipulation Language) arbeitet mit den eigentlichen Daten: `SELECT`, `INSERT`, `UPDATE`, `DELETE`.

> [!NOTE]
> **SQL** ist eine deklarative Sprache. Du beschreibst, **was** du möchtest (alle Produkte mit Gewicht > 2 kg), nicht **wie** die Datenbank das intern umsetzt. Der **Query-Planner** der Datenbank wählt den schnellsten Weg selbst.

Die Groß-/Kleinschreibung der SQL-Schlüsselwörter ist im Standard **nicht** relevant, `SELECT` und `select` funktionieren gleich. Üblich ist die Konvention, SQL-Schlüsselwörter **groß** und eigene Namen (Tabellen, Spalten) in **Unterstrich-** oder **PascalCase**-Schreibweise zu schreiben, damit Queries auf einen Blick lesbar bleiben.


## 5. `CREATE TABLE` – Tabelle anlegen

Mit `CREATE TABLE` legst du eine neue Tabelle an. Für jede Spalte gibst du einen **Namen**, einen **Datentyp** und optional **Constraints** (Bedingungen) an. SQLite kennt nur eine Handvoll Typen: `INTEGER` für ganze Zahlen, `REAL` für Fließkommazahlen, `TEXT` für Zeichenketten, `BLOB` für rohe Binärdaten und `NULL` für den „fehlenden Wert".


In [3]:
# Beispiel-SQL (noch nicht ausgefuehrt, nur zum Anschauen)
sql = '''
CREATE TABLE Produkte (
    Produkt_ID               INTEGER PRIMARY KEY,
    Produktname              TEXT    NOT NULL UNIQUE,
    Produktionszeit_Minuten  INTEGER CHECK (Produktionszeit_Minuten > 0),
    Material_pro_Stueck_kg   REAL    DEFAULT 0.0
)
'''
print(sql)



CREATE TABLE Produkte (
    Produkt_ID               INTEGER PRIMARY KEY,
    Produktname              TEXT    NOT NULL UNIQUE,
    Produktionszeit_Minuten  INTEGER CHECK (Produktionszeit_Minuten > 0),
    Material_pro_Stueck_kg   REAL    DEFAULT 0.0
)



### Die wichtigsten Constraints

`PRIMARY KEY` markiert die eindeutige Identität jeder Zeile. `NOT NULL` verbietet einen fehlenden Wert. `UNIQUE` erlaubt jeden Wert nur einmal. `CHECK (<Bedingung>)` prüft beim Einfügen oder Ändern, ob die Bedingung erfüllt ist. `DEFAULT <wert>` setzt einen Standardwert, wenn beim Einfügen kein Wert für diese Spalte angegeben wurde. Für automatisch hochzählende Primärschlüssel reicht in SQLite `INTEGER PRIMARY KEY` – das Schlüsselwort `AUTOINCREMENT` ist nur in Sonderfällen nötig und kostet Performance.

> [!WARNING]
> `TEXT` ohne `NOT NULL` erlaubt, dass in dieser Spalte `NULL` stehen kann. `NULL` verhält sich anders als der leere String `""` und ist **nicht** gleich `NULL`. Der Vergleich `wert = NULL` liefert immer `NULL` (nicht `TRUE`), korrekt ist `wert IS NULL`.


## 6. `INSERT INTO` – Daten einfügen

Ein `INSERT`-Befehl fügt eine oder mehrere Zeilen in eine Tabelle ein. Die einfachste Form listet erst die Spalten, dann die passenden Werte auf.

```sql
INSERT INTO Produkte (Produkt_ID, Produktname, Produktionszeit_Minuten, Material_pro_Stueck_kg)
VALUES (1, 'Zahnrad Z42', 25, 1.8);
```

Lässt du eine Spalte weg, für die es einen `DEFAULT` oder eine `AUTOINCREMENT`-Regel gibt, wählt die Datenbank den Wert selbst. Strings stehen in **einfachen** Anführungszeichen, Zahlen ohne.


### Mehrere Zeilen auf einmal

Du kannst in einem einzigen `INSERT` auch mehrere Zeilen durch Kommas getrennt einfügen, was bei großen Daten-Imports deutlich schneller ist als viele einzelne Inserts.

```sql
INSERT INTO Produkte (Produkt_ID, Produktname, Produktionszeit_Minuten, Material_pro_Stueck_kg) VALUES
    (2, 'Welle W-18', 35, 3.2),
    (3, 'Flansch F-90', 18, 2.5),
    (4, 'Buchse B-25', 12, 0.8);
```

In Python verwendet man dafür bevorzugt `cursor.executemany(sql, seq_of_params)`, wobei `seq_of_params` eine Liste von Tupeln ist. Dazu kommen wir im Praxis-Notebook.


## 7. `SELECT` – Daten lesen

`SELECT` ist der vielseitigste und am häufigsten verwendete SQL-Befehl. Die Grundform lautet:

```sql
SELECT <spalten>
FROM   <tabelle>
WHERE  <bedingung>
ORDER BY <spalte> [ASC|DESC]
LIMIT  <anzahl>;
```

`SELECT *` liefert alle Spalten – in Produktionscode solltest du die benötigten Spalten immer **explizit** aufzählen, weil sich sonst bei späteren Schema-Änderungen unerwartete Seiteneffekte ergeben.


### Beispiele

```sql
-- Alle Produktnamen
SELECT Produktname FROM Produkte;

-- Alle Produkte mit mehr als 2 kg Material, sortiert nach Gewicht absteigend
SELECT Produkt_ID, Produktname, Material_pro_Stueck_kg
FROM   Produkte
WHERE  Material_pro_Stueck_kg > 2.0
ORDER BY Material_pro_Stueck_kg DESC;

-- Die zwei schnellsten Produkte (kleinste Produktionszeit)
SELECT Produktname, Produktionszeit_Minuten
FROM   Produkte
ORDER BY Produktionszeit_Minuten ASC
LIMIT 2;
```

`WHERE` akzeptiert die üblichen Vergleichsoperatoren (`=`, `!=`, `<`, `>`, `<=`, `>=`) sowie `AND`, `OR`, `NOT`, `IN (...)`, `BETWEEN ... AND ...` und `LIKE '%muster%'`. Wichtig: Zeichenvergleiche verwenden in SQL **ein** Gleichheitszeichen (`=`), nicht `==` wie in Python.


## 8. `UPDATE` und `DELETE`

Mit `UPDATE` änderst du bestehende Zeilen, mit `DELETE` löschst du sie. Beide Befehle haben eine `WHERE`-Klausel – vergisst du diese, werden **alle** Zeilen verändert bzw. gelöscht.

```sql
UPDATE Produkte
SET    Produktionszeit_Minuten = Produktionszeit_Minuten + 5
WHERE  Produkt_ID = 3;

DELETE FROM Produkte WHERE Produkt_ID = 6;
```

> [!WARNING]
> Vergiss niemals die **`WHERE`-Klausel**! Ein `UPDATE Produkte SET Produktionszeit_Minuten = 0;` ohne `WHERE` setzt die Zeit **aller** Produkte auf 0. Ein `DELETE FROM Produkte;` leert die komplette Tabelle. Beides ist ohne Backup nicht mehr umkehrbar.

Zum Üben kannst du in SQLite vor einem heiklen `UPDATE` eine **Transaktion** starten (`BEGIN`) und im Fehlerfall mit `ROLLBACK` alles zurückdrehen. Dazu gleich mehr.


## 9. Transaktionen – zwei Zeilen über Commit

Jede Änderung (`INSERT`, `UPDATE`, `DELETE`) läuft in SQLite innerhalb einer **Transaktion**. Die Änderungen sind erst nach einem `COMMIT` dauerhaft gespeichert; bis dahin sieht sie nur deine eigene Verbindung. Das Gegenteil ist `ROLLBACK`: Alle seit Transaktionsbeginn ausgeführten Änderungen werden verworfen.

In Python macht das `sqlite3`-Modul das komfortabel: Nach einer ändernden Operation rufst du `conn.commit()` auf, wenn alles in Ordnung war, oder `conn.rollback()` im Fehlerfall. Ein `with conn:`-Block übernimmt das automatisch.

> [!NOTE]
> Eine **Transaktion** ist eine logische Einheit von Datenbank-Operationen, die entweder vollständig (`COMMIT`) oder gar nicht (`ROLLBACK`) angewendet wird. Dieses **Alles-oder-Nichts**-Prinzip ist eine zentrale Eigenschaft zuverlässiger Datenbanken (das **A** in **ACID**).


## 10. Das Python-Modul `sqlite3`

Python liefert in der Standardbibliothek das Modul `sqlite3` mit. Damit sind vier Objekte wichtig: die **Connection**, der **Cursor**, einzelne **Queries** mit `execute` und das **Ergebnis-Objekt** mit `fetchone`/`fetchall`.


In [4]:
# Muster-Code zum Ansehen (wird im Praxis-Notebook ausgefuehrt)
beispiel = '''
import sqlite3

with sqlite3.connect(":memory:") as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, name TEXT)")
    cursor.execute("INSERT INTO t (name) VALUES (?)", ("Alice",))
    cursor.execute("INSERT INTO t (name) VALUES (?)", ("Bob",))
    for zeile in cursor.execute("SELECT id, name FROM t ORDER BY id"):
        id_, name = zeile
        print(id_, name)
'''
print(beispiel)



import sqlite3

with sqlite3.connect(":memory:") as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, name TEXT)")
    cursor.execute("INSERT INTO t (name) VALUES (?)", ("Alice",))
    cursor.execute("INSERT INTO t (name) VALUES (?)", ("Bob",))
    for zeile in cursor.execute("SELECT id, name FROM t ORDER BY id"):
        id_, name = zeile
        print(id_, name)



### `connect`, `cursor`, `execute`

`sqlite3.connect(pfad)` öffnet (oder erzeugt) eine Datenbank-Datei und liefert eine **Connection**. `:memory:` erzeugt eine reine RAM-Datenbank. Auf der Connection rufst du `cursor()` auf und erhältst einen **Cursor**, der deine SQL-Befehle entgegennimmt. Mit `cursor.execute(sql)` oder `cursor.execute(sql, parameter)` läuft eine Query; mit `cursor.executemany(sql, liste)` gleich viele Inserts auf einmal.

Lese-Queries liefern den Cursor selbst zurück, über den du entweder iterieren kannst (`for row in cursor.execute(...)`) oder die Zeilen explizit mit `fetchall()` (alle als Liste) bzw. `fetchone()` (nächste einzelne Zeile) abholst.


## 11. Parameter-Binding und SQL-Injection

Kommen wir zur **wichtigsten Regel** dieser Vorlesung: Nutzereingaben dürfen **niemals** per String-Konkatenation in eine SQL-Query eingebaut werden. Das öffnet Angreifern Tür und Tor zu einer **SQL-Injection**.

> [!WARNING]
> **SQL-Injection** ist eine der häufigsten Sicherheitslücken überhaupt. Der Angreifer schleust über ein Eingabefeld Teile einer SQL-Query ein und übernimmt so die Kontrolle über deine Datenbank – bis hin zu kompletter Datenlöschung.

Die einzige korrekte Methode ist das **Parameter-Binding**. Du schreibst in die Query ein Fragezeichen `?` als Platzhalter und übergibst die Werte als Tupel in einem zweiten Argument an `execute`. Der Datenbank-Treiber kümmert sich um das sichere Escaping.


In [5]:
# Diagramm: SQL-Injection vs. Parameter-Binding
from IPython.display import Markdown, display
with open("diagramme/03_sql_injection.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart LR
    subgraph Unsicher["Unsicher: String-Konkatenation"]
        U1["Eingabe: ' OR '1'='1"] --> U2["SQL: SELECT ... WHERE name='' OR '1'='1'"]
        U2 --> U3["Alle Zeilen werden zurueckgegeben"]
    end
    subgraph Sicher["Sicher: Parameter-Binding"]
        S1["Eingabe: ' OR '1'='1"] --> S2["cursor.execute('... WHERE name=?', (eingabe,))"]
        S2 --> S3["Treiber escaped automatisch"]
    end

```

In [6]:
# Demonstration (nur zur Anschauung, nicht anwenden)
unsicher = '''
eingabe = "' OR '1'='1"   # Angreifer-Eingabe
query   = "SELECT * FROM Produkte WHERE Produktname = '" + eingabe + "'"
# -> SELECT * FROM Produkte WHERE Produktname = '' OR '1'='1'
# -> Datenbank liefert ALLE Zeilen zurueck!
'''
sicher = '''
eingabe = "' OR '1'='1"
cursor.execute("SELECT * FROM Produkte WHERE Produktname = ?", (eingabe,))
# -> Datenbank sucht wirklich nach dem buchstaeblichen String "' OR '1'='1"
# -> Ergebnis: leer. Angriff ins Leere gelaufen.
'''
print("Unsicher:\n", unsicher)
print("Sicher:\n", sicher)


Unsicher:
 
eingabe = "' OR '1'='1"   # Angreifer-Eingabe
query   = "SELECT * FROM Produkte WHERE Produktname = '" + eingabe + "'"
# -> SELECT * FROM Produkte WHERE Produktname = '' OR '1'='1'
# -> Datenbank liefert ALLE Zeilen zurueck!

Sicher:
 
eingabe = "' OR '1'='1"
cursor.execute("SELECT * FROM Produkte WHERE Produktname = ?", (eingabe,))
# -> Datenbank sucht wirklich nach dem buchstaeblichen String "' OR '1'='1"
# -> Ergebnis: leer. Angriff ins Leere gelaufen.



> [!TIP]
> **Merksatz:** Niemals SQL per `+`, `%` oder f-String zusammenbauen, sobald **irgendein** Wert aus Nutzereingabe, Datei oder Netz stammt. Immer `?`-Platzhalter und Tupel-Argument verwenden. Das kostet keine Zeile Code mehr und schützt dich und deine Nutzer.


## 12. CRUD im Überblick

Die vier grundlegenden Operationen auf Daten heißen oft nach ihren englischen Anfangsbuchstaben **CRUD**: **Create**, **Read**, **Update**, **Delete**. Jede fachliche Anforderung an eine Datenbank lässt sich letztlich auf diese vier Operationen (plus gezielte Abfragen mit `WHERE`/`ORDER BY`) zurückführen.


In [7]:
# Diagramm: CRUD-Operationen
from IPython.display import Markdown, display
with open("diagramme/02_crud_operationen.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart LR
    C["CREATE TABLE\n(Struktur anlegen)"] --> I["INSERT INTO\n(Daten einfuegen)"]
    I --> S["SELECT\n(Daten lesen)"]
    S --> U["UPDATE\n(Daten aendern)"]
    U --> D["DELETE\n(Daten loeschen)"]
    D --> S

```

## 13. Maschinenbau-Beispiel: Eine Produktionsdatenbank

Damit die Theorie nicht abstrakt bleibt, denken wir uns eine kleine **Produktionsdatenbank** aus. Wir haben Tabellen für **Maschinen** (Drehmaschinen, Fräsen, 3D-Drucker), für **Produkte** (Zahnrad, Welle, Flansch) und für **Wartungen** (Wer? Wann? Was wurde gemacht?). Eine typische Frage an diese Datenbank wäre: „Welche Maschine hat im letzten Monat die meisten Werkstücke produziert?" oder „Welche Maschinen haben seit über 500 Betriebsstunden keine Wartung mehr gesehen?".

In V19 beschränken wir uns bewusst auf **eine einzelne Tabelle** (unsere `Produkte`-Tabelle aus `produkte.csv`), damit wir den CRUD-Kern sauber üben können. Die Welt der Joins, Fremdschlüssel-Constraints und Normalformen folgt in V20.


### Vorschau: Tabellen-Schema der Übungsdaten

Die Datei `testdaten/produkte.csv` enthält sechs Produkte mit Produktionszeit und Materialbedarf. Die Tabelle, die wir gleich anlegen, sieht so aus:

| Spalte | Typ | Constraint | Bedeutung |
|---|---|---|---|
| `Produkt_ID` | `INTEGER` | `PRIMARY KEY` | Eindeutige Identität |
| `Produktname` | `TEXT` | `NOT NULL` | Bezeichnung des Produkts |
| `Produktionszeit_Minuten` | `INTEGER` | – | Zeit pro Stück in Minuten |
| `Material_pro_Stueck_kg` | `REAL` | – | Materialbedarf in kg |


## 14. Mini-Demo: SQL in wenigen Zeilen

Die folgende Zelle zeigt das komplette CRUD-Spektrum in etwa zehn Zeilen Code. Lies sie entspannt, jede Zeile dürfte dir mit dem bisherigen Wissen sofort einleuchten.


In [8]:
import sqlite3

with sqlite3.connect(":memory:") as conn:
    cur = conn.cursor()

    # CREATE
    cur.execute('''CREATE TABLE Produkte (
        Produkt_ID INTEGER PRIMARY KEY,
        Produktname TEXT NOT NULL,
        Produktionszeit_Minuten INTEGER,
        Material_pro_Stueck_kg REAL
    )''')

    # INSERT (parametrisiert!)
    cur.execute("INSERT INTO Produkte VALUES (?, ?, ?, ?)", (1, "Zahnrad Z42", 25, 1.8))
    cur.execute("INSERT INTO Produkte VALUES (?, ?, ?, ?)", (2, "Welle W-18", 35, 3.2))

    # READ
    print("Alle Produkte:")
    for pid, name, zeit, masse in cur.execute("SELECT * FROM Produkte"):
        print(f"  {pid} {name} - {zeit} min, {masse} kg")

    # UPDATE
    cur.execute("UPDATE Produkte SET Produktionszeit_Minuten = ? WHERE Produkt_ID = ?", (30, 1))

    # DELETE
    cur.execute("DELETE FROM Produkte WHERE Produkt_ID = ?", (2,))

    # Finales SELECT
    print("\nNach Update + Delete:")
    for row in cur.execute("SELECT * FROM Produkte"):
        print(" ", row)


Alle Produkte:
  1 Zahnrad Z42 - 25 min, 1.8 kg
  2 Welle W-18 - 35 min, 3.2 kg

Nach Update + Delete:
  (1, 'Zahnrad Z42', 30, 1.8)


## 15. Selbst-Check – kurze Verständnis-Fragen

<details>
<summary>① Wodurch unterscheidet sich eine Datenbank von einer CSV-Datei?</summary>

Eine Datenbank erzwingt ein **festes Schema**, bietet **Integritätsbedingungen** wie `PRIMARY KEY`, erlaubt **ausdrucksstarke Abfragen** per SQL und ermöglicht **gleichzeitigen Zugriff** durch mehrere Programme. Eine CSV-Datei ist dagegen nur eine unstrukturierte Text-Datei ohne jede dieser Garantien.
</details>

<details>
<summary>② Was ist ein Primärschlüssel, was ein Fremdschlüssel?</summary>

Ein **Primärschlüssel** identifiziert jede Zeile einer Tabelle eindeutig. Ein **Fremdschlüssel** ist eine Spalte, deren Werte auf die Primärschlüssel-Werte einer anderen Tabelle verweisen und so eine Beziehung herstellen.
</details>

<details>
<summary>③ Warum ist `SELECT * FROM ...` in produktivem Code problematisch?</summary>

Sobald die Tabelle um Spalten erweitert oder reduziert wird, ändert sich stillschweigend auch das Ergebnis der Query. Code, der auf eine bestimmte Reihenfolge oder Anzahl der Spalten vertraut, bricht plötzlich oder liefert falsche Daten. Besser: die benötigten Spalten **explizit** auflisten.
</details>

<details>
<summary>④ Was passiert bei einem `UPDATE` ohne `WHERE`-Klausel?</summary>

Ohne `WHERE` gilt das `UPDATE` für **alle** Zeilen der Tabelle. Jede Zeile wird mit den angegebenen Werten überschrieben. Ohne Backup ist das in der Regel nicht rückgängig zu machen.
</details>

<details>
<summary>⑤ Wie heißt die einzige sichere Methode, Nutzereingaben in eine SQL-Query einzubauen?</summary>

**Parameter-Binding** mit `?`-Platzhaltern. Die Werte werden als Tupel im zweiten Argument von `cursor.execute(sql, params)` übergeben und vom Datenbank-Treiber sicher escaped. Nie per String-Konkatenation, f-String oder `%`-Operator zusammenbauen.
</details>

<details>
<summary>⑥ Wann solltest du SQLite statt PostgreSQL wählen?</summary>

Für **eingebettete** Anwendungen, **Lernumgebungen**, **Desktop-Tools** und **Prototypen** mit einem einzelnen oder wenigen gleichzeitigen Nutzern. Sobald viele gleichzeitige Schreiber, Netzwerk-Zugriff oder feingranulare Rechte nötig sind, ist ein Server-DBMS wie PostgreSQL die bessere Wahl.
</details>

<details>
<summary>⑦ Was bedeutet `conn.commit()`?</summary>

Alle seit dem letzten Commit ausgeführten Änderungen werden **dauerhaft** in die Datenbank geschrieben. Ohne Commit bleiben sie innerhalb der aktuellen Transaktion und sind für andere Verbindungen unsichtbar. Bei In-Memory-Datenbanken ist das Commit oft verzichtbar, bei Datei-DBs ist es Pflicht, damit die Änderungen den Prozess überleben.
</details>

<details>
<summary>⑧ Welche fünf SQL-Befehle bilden den Kern dieser Vorlesung?</summary>

`CREATE TABLE`, `INSERT`, `SELECT`, `UPDATE` und `DELETE`. Zusammen bilden sie die **CRUD**-Operationen (Create, Read, Update, Delete).
</details>


## ✅ Zusammenfassung
- Eine **relationale Datenbank** organisiert Daten in Tabellen mit festen Spalten, Typen und Integritätsbedingungen.
- **SQLite** ist eine Datei-basierte Embedded-Datenbank, ideal für Lernen, Prototypen und kleine Tools.
- Die **fünf SQL-Kernbefehle** sind `CREATE TABLE`, `INSERT`, `SELECT`, `UPDATE`, `DELETE`.
- **Parameter-Binding** mit `?` ist die einzige sichere Methode gegen **SQL-Injection**.
- `sqlite3.connect` → `cursor()` → `execute(sql, params)` → `fetchall()`/`fetchone()` ist das Grundmuster in Python.

## ➡️ Nächster Schritt
Weiter mit [02_praxis.ipynb](02_praxis.ipynb) – dort setzt du alles an einer In-Memory-Datenbank um.
